# Stream checkpoint assessment

Load a trained **CPEN** stream run (`best.ckpt`), score every MOCK graph, sweep the
**discovery threshold**, and draw streams in $(\phi,\lambda)$ at the chosen operating point.

**Metric roles**

| Kind | Examples | Role |
|---|---|---|
| Summary (threshold-free) | AUROC, PR-AUC | Ranking quality |
| Operating-point | precision, recall, TP/FP/FN/TN, F1 | What a cut actually does |
| Threshold choice | max F1 / max F$_\beta$ / P@$R\ge R_\min$ | Pick $t^\star$ for maps |

Retarget `RUN_DIR` after a new Slurm job.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib.collections import LineCollection
from sklearn.metrics import average_precision_score, roc_auc_score

REPO = Path("..").resolve()
sys.path[:0] = [str(REPO), str(REPO / "src")]

from cpen.lit_models.lit_cpen_stream import LitCPENStream
from cpen.models.cpen import CPEN
from cpen.utils.stream_graph_cache import (
    DEFAULT_STREAM_DATA_DIR,
    N_EDGE_FEATURES,
    N_FEATURES,
    OUT_DIM,
    list_mock_stream_names,
    load_stream_pyg,
    pyg_to_incidence_payload,
    resolve_stream_graph_path,
)

import scienceplots  # noqa: F401

plt.style.use(["science", "no-latex"])
mpl.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
    }
)

%matplotlib inline

DATA_DIR = Path(DEFAULT_STREAM_DATA_DIR)
# 12588634: product-consistency edge aux, η=0.05, t=3, drop=0.4, monitor=val_auroc
RUN_DIR = Path(
    "/scratch/gpfs/BHANIN/jdezoort/cpen_runs/stream_output/adamw/sweep_lr/"
    "cpen_4_256_0p05_t3_ln_op-incidence_onorm-degree_res-transformer-like_drop0p4_nf7_gnativeedges_edgecons"
)
CKPT = RUN_DIR / "best.ckpt"
PARQUET = RUN_DIR.with_suffix(".parquet")
if not PARQUET.is_file():
    PARQUET = RUN_DIR.parent / f"{RUN_DIR.name}.parquet"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STREAMS_TO_SCORE: list[str] | None = None  # None = every MOCK
STREAMS_TO_DRAW = ['MOCK_1223', 'MOCK_1368', 'MOCK_1762', 'MOCK_1863', 'MOCK_2041', 'MOCK_2067', 'MOCK_2828', 'MOCK_2941', 'MOCK_3095', 'MOCK_4087', 'MOCK_440', 'MOCK_4405', 'MOCK_456', 'MOCK_4626', 'MOCK_613']

# Discovery threshold selection on the pooled ~train_mask stars.
#   "f1"      — maximize F1
#   "fbeta"   — maximize F_beta (beta>1 favors recall)
#   "prec_at_recall" — max precision subject to recall >= MIN_RECALL
THRESH_RULE = "f1"
FBETA = 2.0
MIN_RECALL = 0.5
# Optimal cuts live near ~1 for these runs — dense high-t grid, sparse below.
THRESH_GRID = np.unique(
    np.concatenate(
        [
            np.linspace(0.50, 0.90, 9),       # coarse context
            np.linspace(0.90, 0.99, 46),      # ~0.002 steps
            np.linspace(0.990, 0.999, 19),    # fine near 1
            np.array([0.95, 0.97, 0.98, 0.99, 0.995, 0.999]),
        ]
    )
)

FIG_DIR = Path("figures/stream_checkpoint_viz")
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.is_dir(), DATA_DIR
assert CKPT.is_file(), CKPT
print(f"data: {DATA_DIR}")
print(f"ckpt: {CKPT}")
print(f"parquet: {PARQUET} exists={PARQUET.is_file()}")
print(f"device: {DEVICE}")
print(f"threshold rule: {THRESH_RULE}")
print(f"mocks available: {list_mock_stream_names(DATA_DIR)}")

## Training curves (node vs edge over time)

In [ ]:
def epoch_frame(hist: pd.DataFrame) -> pd.DataFrame:
    if "record_type" in hist.columns:
        ep = hist.loc[hist["record_type"] == "epoch"].copy()
    else:
        ep = hist.copy()
    return ep.sort_values("epoch").reset_index(drop=True)


if PARQUET.is_file():
    hist = pd.read_parquet(PARQUET)
    epochs = epoch_frame(hist)
    edge_cols = [c for c in ("val_edge_auroc", "train_edge_loss", "val_edge_loss") if c in epochs]
    show = [
        c
        for c in ("epoch", "train_loss", "val_loss", "train_acc", "val_acc", "val_auroc", *edge_cols)
        if c in epochs
    ]
    print(epochs[show].tail())

    fig, axes = plt.subplots(2, 2, figsize=(9.0, 5.6), sharex=True)

    ax = axes[0, 0]
    if "train_node_loss" in epochs and "train_edge_loss" in epochs:
        ax.plot(epochs["epoch"], epochs["train_node_loss"], label="train node")
        ax.plot(epochs["epoch"], epochs["train_edge_loss"], label="train edge")
        if "val_node_loss" in epochs:
            ax.plot(epochs["epoch"], epochs["val_node_loss"], ls="--", label="val node")
        if "val_edge_loss" in epochs:
            ax.plot(epochs["epoch"], epochs["val_edge_loss"], ls="--", label="val edge")
    else:
        ax.plot(epochs["epoch"], epochs["train_loss"], label="train")
        ax.plot(epochs["epoch"], epochs["val_loss"], label="val")
    ax.set_yscale("log")
    ax.set_ylabel("CE loss")
    ax.set_title("Loss (node vs edge)")
    ax.legend(loc="best")

    ax = axes[0, 1]
    ax.plot(epochs["epoch"], epochs["train_acc"], label="train node")
    ax.plot(epochs["epoch"], epochs["val_acc"], label="val node")
    if "train_edge_acc" in epochs:
        ax.plot(epochs["epoch"], epochs["train_edge_acc"], label="train edge")
    if "val_edge_acc" in epochs:
        ax.plot(epochs["epoch"], epochs["val_edge_acc"], label="val edge")
    ax.set_ylabel("accuracy")
    ax.set_title("Supervised-pool accuracy")
    ax.legend(loc="best")

    ax = axes[1, 0]
    if "val_auroc" in epochs:
        ax.plot(epochs["epoch"], epochs["val_auroc"], color="0.35", label="node val_auroc")
    if "val_edge_auroc" in epochs:
        ax.plot(
            epochs["epoch"],
            epochs["val_edge_auroc"],
            color="C2",
            marker="o",
            ms=3,
            label="edge val_edge_auroc",
        )
    ax.axhline(0.5, color="0.7", lw=0.8, ls=":")
    ax.set_ylim(0.45, 1.02)
    ax.set_xlabel("epoch")
    ax.set_ylabel("discovery AUROC")
    ax.set_title("Discovery AUROC vs train time")
    ax.legend(loc="best")

    ax = axes[1, 1]
    if {"train_edge_loss", "val_edge_loss", "val_edge_auroc"} <= set(epochs.columns):
        ax.plot(epochs["epoch"], epochs["train_edge_loss"], label="train edge CE")
        ax.plot(epochs["epoch"], epochs["val_edge_loss"], label="val edge CE")
        ax.set_yscale("log")
        ax.set_ylabel("edge CE")
        ax2 = ax.twinx()
        ax2.plot(
            epochs["epoch"],
            epochs["val_edge_auroc"],
            color="C2",
            marker="o",
            ms=3,
            label="val_edge_auroc",
        )
        ax2.set_ylabel("edge discovery AUROC")
        ax2.set_ylim(0.45, 1.02)
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines + lines2, labels + labels2, loc="best")
        ax.set_title("Edge diagnostics")
    else:
        ax.text(0.5, 0.5, "no edge metric columns", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("epoch")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "training_curves_node_edge.pdf")
    plt.show()
else:
    hist = None
    print(f"No parquet at {PARQUET}; skip curves.")

## Load `best.ckpt`

In [ ]:
def run_hparams(hist: pd.DataFrame | None) -> dict:
    defaults = dict(
        depth=4,
        width=256,
        eta_0=0.01,
        residual_structure="transformer-like",
        operator_normalization="degree",
        layer_norm=True,
        dropout=0.4,
    )
    if hist is None or hist.empty:
        return defaults
    row = hist.iloc[-1]

    def _get(key, default):
        if key not in row.index or pd.isna(row[key]):
            return default
        return row[key]

    return dict(
        depth=int(_get("depth", defaults["depth"])),
        width=int(_get("width", defaults["width"])),
        eta_0=float(_get("eta_0", defaults["eta_0"])),
        residual_structure=str(_get("residual_structure", defaults["residual_structure"])),
        operator_normalization=str(
            _get("operator_normalization", defaults["operator_normalization"])
        ),
        layer_norm=bool(_get("layer_norm", defaults["layer_norm"])),
        dropout=float(_get("dropout", defaults["dropout"]) or 0.0),
    )


hp = run_hparams(hist)
print(hp)

model = CPEN(
    n_features=N_FEATURES,
    n_edge_features=N_EDGE_FEATURES,
    out_dim=OUT_DIM,
    depth=hp["depth"],
    width=hp["width"],
    operators="incidence",
    normalization="uniform",
    residual_structure=hp["residual_structure"],
    operator_normalization=hp["operator_normalization"],
    operator_backend="sparse",
    layer_norm=hp["layer_norm"],
    dropout=hp["dropout"],
    readout_mode="node+edge",
)
lit = LitCPENStream.load_from_checkpoint(
    str(CKPT),
    model=model,
    eta_0=hp["eta_0"],
    model_name="cpen",
    optimizer="adamw",
    dataset="stream",
    depth=hp["depth"],
    width=hp["width"],
    map_location="cpu",
    strict=False,
)
lit.eval()
lit.to(DEVICE)
print(f"loaded {CKPT.name} on {DEVICE}")

## Score all MOCK streams

One forward per graph. Store scores; threshold-dependent metrics come next.

In [ ]:
def batch_from_payload(payload: dict[str, torch.Tensor], device: torch.device) -> dict:
    keys = [
        "x",
        "edge_x",
        "incidence_node",
        "incidence_edge",
        "incidence_nnz",
        "node_degree_inv",
        "edge_degree_inv",
        "mask",
    ]
    return {k: payload[k].unsqueeze(0).to(device) for k in keys}


def _auroc(score: np.ndarray, target: np.ndarray, mask: np.ndarray) -> float:
    m = mask.astype(bool)
    y = target[m].astype(int)
    s = score[m]
    if y.size == 0 or len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, s))


def _ap(score: np.ndarray, target: np.ndarray, mask: np.ndarray) -> float:
    m = mask.astype(bool)
    y = target[m].astype(int)
    s = score[m]
    if y.size == 0 or y.sum() == 0:
        return float("nan")
    return float(average_precision_score(y, s))


@torch.no_grad()
def score_stream(name: str) -> dict:
    path = resolve_stream_graph_path(DATA_DIR, stream_name=name)
    raw = load_stream_pyg(path)
    payload = pyg_to_incidence_payload(raw)
    batch = batch_from_payload(payload, DEVICE)
    node_logits, edge_logits = lit.model(batch["x"], **lit._model_kwargs(batch))
    p_node = F.softmax(node_logits[0], dim=-1)[:, 1].detach().cpu().numpy()
    p_edge = F.softmax(edge_logits[0], dim=-1)[:, 1].detach().cpu().numpy()

    y = payload["y"].cpu().numpy().astype(bool)
    ey = payload["edge_y"].cpu().numpy().astype(bool)
    tr = payload["train_mask"].cpu().numpy().astype(bool)
    va = payload["val_mask"].cpu().numpy().astype(bool)
    te = payload["test_mask"].cpu().numpy().astype(bool)
    etr = payload["edge_train_mask"].cpu().numpy().astype(bool)
    pool = ~tr
    epool = ~etr
    holdout = va | te
    pairs = np.stack(
        [
            payload["incidence_node"][0::2].cpu().numpy(),
            payload["incidence_node"][1::2].cpu().numpy(),
        ],
        axis=0,
    )
    p_prod = p_node[pairs[0]] * p_node[pairs[1]]

    return dict(
        name=name,
        stream_label=str(getattr(raw, "stream_label", name)),
        n_nodes=int(y.size),
        n_edges=int(ey.size),
        y_pos=int(y.sum()),
        edge_y_pos=int(ey.sum()),
        discovery_pos=int((pool & y).sum()),
        discovery_neg=int((pool & ~y).sum()),
        discovery_pos_frac=float((pool & y).sum() / max(int(pool.sum()), 1)),
        discovery_auroc=_auroc(p_node, y, pool),
        discovery_ap=_ap(p_node, y, pool),
        edge_discovery_auroc=_auroc(p_edge, ey, epool),
        edge_discovery_ap=_ap(p_edge, ey, epool),
        edge_product_auroc=_auroc(p_prod, ey, epool),
        mean_p_y1=float(p_node[y].mean()) if y.any() else float("nan"),
        mean_p_y0=float(p_node[~y].mean()) if (~y).any() else float("nan"),
        p_node=p_node,
        p_edge=p_edge,
        p_prod=p_prod,
        y=y,
        edge_y=ey,
        train_mask=tr,
        val_mask=va,
        test_mask=te,
        holdout_mask=holdout,
        pool=pool,
        epool=epool,
        phi=raw.x[:, 0].numpy(),
        lam=raw.x[:, 1].numpy(),
        pairs=pairs,
    )


names = STREAMS_TO_SCORE or list_mock_stream_names(DATA_DIR)
scored: dict[str, dict] = {}
rows = []
for i, name in enumerate(names):
    print(f"[{i+1}/{len(names)}] {name}", flush=True)
    result = score_stream(name)
    scored[name] = result
    rows.append({k: result[k] for k in result if not isinstance(result[k], np.ndarray)})

summary = pd.DataFrame(rows).sort_values("discovery_auroc", ascending=False)
display(
    summary[
        [
            "name",
            "discovery_pos",
            "discovery_pos_frac",
            "discovery_auroc",
            "discovery_ap",
            "edge_discovery_auroc",
            "edge_product_auroc",
            "mean_p_y1",
            "mean_p_y0",
        ]
    ].round(4)
)
print(
    "macro node AUROC / AP:",
    f"{summary['discovery_auroc'].mean():.4f} / {summary['discovery_ap'].mean():.4f}",
)
print(
    "macro edge AUROC (learned / product):",
    f"{summary['edge_discovery_auroc'].mean():.4f} / {summary['edge_product_auroc'].mean():.4f}",
)

## Threshold sweep → choose $t^\star$

Pool all discovery-pool stars across MOCKS. Sweep $t$, record confusion-matrix entries and
P/R/F1. Pick $t^\star$ with `THRESH_RULE`, then re-evaluate **per MOCK** at that cut.

AUROC / AP above stay the ranking summaries; everything below is operating-point.

In [ ]:
def confusion_at_threshold(
    score: np.ndarray, target: np.ndarray, mask: np.ndarray, thresh: float
) -> dict[str, float]:
    m = mask.astype(bool)
    y = target[m].astype(bool)
    pred = score[m] >= thresh
    tp = int((pred & y).sum())
    fp = int((pred & ~y).sum())
    fn = int((~pred & y).sum())
    tn = int((~pred & ~y).sum())
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    rec = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    f1 = (
        2 * prec * rec / (prec + rec)
        if np.isfinite(prec) and np.isfinite(rec) and (prec + rec) > 0
        else float("nan")
    )
    fbeta = (
        (1 + FBETA**2) * prec * rec / (FBETA**2 * prec + rec)
        if np.isfinite(prec) and np.isfinite(rec) and (FBETA**2 * prec + rec) > 0
        else float("nan")
    )
    return dict(
        threshold=float(thresh),
        tp=tp,
        fp=fp,
        fn=fn,
        tn=tn,
        precision=prec,
        recall=rec,
        specificity=spec,
        f1=f1,
        fbeta=fbeta,
        n_pred=int(pred.sum()),
        n_pos=int(y.sum()),
        n_neg=int((~y).sum()),
    )


def threshold_curve(
    score: np.ndarray,
    target: np.ndarray,
    mask: np.ndarray,
    grid: np.ndarray = THRESH_GRID,
) -> pd.DataFrame:
    return pd.DataFrame([confusion_at_threshold(score, target, mask, t) for t in grid])


def choose_threshold(curve: pd.DataFrame, rule: str = THRESH_RULE) -> pd.Series:
    df = curve.copy()
    if rule == "f1":
        return df.loc[df["f1"].idxmax()]
    if rule == "fbeta":
        return df.loc[df["fbeta"].idxmax()]
    if rule == "prec_at_recall":
        ok = df[df["recall"] >= MIN_RECALL]
        if ok.empty:
            # fall back to highest recall, then best precision among those
            return df.sort_values(["recall", "precision"], ascending=[False, False]).iloc[0]
        return ok.loc[ok["precision"].idxmax()]
    raise ValueError(f"unknown THRESH_RULE={rule!r}")


# ---- pooled node discovery (all MOCKS) ----
p_all = np.concatenate([scored[n]["p_node"][scored[n]["pool"]] for n in scored])
y_all = np.concatenate([scored[n]["y"][scored[n]["pool"]] for n in scored])
mask_all = np.ones(p_all.shape[0], dtype=bool)
node_curve = threshold_curve(p_all, y_all, mask_all)
best_node = choose_threshold(node_curve)
OPT_THRESH = float(best_node["threshold"])

# ---- pooled edge discovery ----
pe_all = np.concatenate([scored[n]["p_edge"][scored[n]["epool"]] for n in scored])
ey_all = np.concatenate([scored[n]["edge_y"][scored[n]["epool"]] for n in scored])
edge_curve = threshold_curve(pe_all, ey_all, np.ones(pe_all.shape[0], dtype=bool))
best_edge = choose_threshold(edge_curve)
OPT_EDGE_THRESH = float(best_edge["threshold"])

print(f"pooled discovery positives: {int(y_all.sum())} / {len(y_all)} "
      f"({100*y_all.mean():.3f}%)")
print(f"node t* ({THRESH_RULE}) = {OPT_THRESH:.3f}")
print(
    f"  P={best_node['precision']:.4f}  R={best_node['recall']:.4f}  "
    f"F1={best_node['f1']:.4f}  Fβ={best_node['fbeta']:.4f}"
)
print(
    f"  TP={int(best_node['tp'])}  FP={int(best_node['fp'])}  "
    f"FN={int(best_node['fn'])}  TN={int(best_node['tn'])}"
)
print(f"edge t* ({THRESH_RULE}) = {OPT_EDGE_THRESH:.3f}")
print(
    f"  P={best_edge['precision']:.4f}  R={best_edge['recall']:.4f}  "
    f"TP={int(best_edge['tp'])} FP={int(best_edge['fp'])} FN={int(best_edge['fn'])}"
)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))

ax = axes[0]
ax.plot(node_curve["threshold"], node_curve["precision"], label="precision")
ax.plot(node_curve["threshold"], node_curve["recall"], label="recall")
ax.plot(node_curve["threshold"], node_curve["f1"], label="F1")
ax.plot(node_curve["threshold"], node_curve["fbeta"], ls="--", label=f"F$\\beta$($\\beta$={FBETA:g})")
ax.axvline(OPT_THRESH, color="0.3", ls=":", label=f"t*={OPT_THRESH:.3f}")
ax.set_xlabel("threshold t")
ax.set_ylabel("score")
ax.set_xlim(0.88, 1.001)
ax.set_ylim(0, 1.05)
ax.set_title("Node discovery vs threshold (pooled, high-t zoom)")
ax.legend(loc="best", fontsize=7)

ax = axes[1]
ax.plot(node_curve["threshold"], node_curve["tp"], label="TP")
ax.plot(node_curve["threshold"], node_curve["fp"], label="FP")
ax.plot(node_curve["threshold"], node_curve["fn"], label="FN")
ax.axvline(OPT_THRESH, color="0.3", ls=":", label=f"t*={OPT_THRESH:.3f}")
ax.set_yscale("log")
ax.set_xlabel("threshold t")
ax.set_ylabel("count")
ax.set_xlim(0.88, 1.001)
ax.set_title("Confusion counts vs threshold")
ax.legend(loc="best", fontsize=7)

fig.tight_layout()
fig.savefig(FIG_DIR / "threshold_sweep_nodes.pdf")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
ax = axes[0]
ax.plot(edge_curve["threshold"], edge_curve["precision"], label="precision")
ax.plot(edge_curve["threshold"], edge_curve["recall"], label="recall")
ax.plot(edge_curve["threshold"], edge_curve["f1"], label="F1")
ax.axvline(OPT_EDGE_THRESH, color="0.3", ls=":", label=f"t*={OPT_EDGE_THRESH:.3f}")
ax.set_xlabel("threshold t")
ax.set_ylabel("score")
ax.set_xlim(0.88, 1.001)
ax.set_ylim(0, 1.05)
ax.set_title("Edge discovery vs threshold (pooled, high-t zoom)")
ax.legend(loc="best", fontsize=7)

ax = axes[1]
ax.plot(edge_curve["threshold"], edge_curve["tp"], label="TP")
ax.plot(edge_curve["threshold"], edge_curve["fp"], label="FP")
ax.plot(edge_curve["threshold"], edge_curve["fn"], label="FN")
ax.axvline(OPT_EDGE_THRESH, color="0.3", ls=":")
ax.set_yscale("log")
ax.set_xlabel("threshold t")
ax.set_ylabel("count")
ax.set_xlim(0.88, 1.001)
ax.set_title("Edge confusion counts vs threshold")
ax.legend(loc="best", fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "threshold_sweep_edges.pdf")
plt.show()

# Reference table focused on the high-t region (+ t*).
show_t = sorted(
    {
        round(float(t), 3)
        for t in np.concatenate(
            [
                np.linspace(0.90, 0.99, 10),
                np.array([0.995, 0.999, OPT_THRESH, OPT_EDGE_THRESH]),
            ]
        )
    }
)
ref = node_curve.copy()
ref["t_round"] = ref["threshold"].round(3)
ref = (
    ref[ref["t_round"].isin(show_t)]
    .drop_duplicates("t_round")
    .sort_values("t_round")
)
print("\nNode pooled metrics at high thresholds:")
display(
    ref[["threshold", "precision", "recall", "f1", "tp", "fp", "fn", "tn"]].round(4)
)

In [ ]:
# Per-MOCK operating point at the pooled t*
op_rows = []
for name, r in scored.items():
    node = confusion_at_threshold(r["p_node"], r["y"], r["pool"], OPT_THRESH)
    edge = confusion_at_threshold(r["p_edge"], r["edge_y"], r["epool"], OPT_EDGE_THRESH)
    op_rows.append(
        dict(
            name=name,
            discovery_auroc=r["discovery_auroc"],
            discovery_ap=r["discovery_ap"],
            thresh=OPT_THRESH,
            precision=node["precision"],
            recall=node["recall"],
            f1=node["f1"],
            tp=node["tp"],
            fp=node["fp"],
            fn=node["fn"],
            tn=node["tn"],
            edge_auroc=r["edge_discovery_auroc"],
            edge_thresh=OPT_EDGE_THRESH,
            edge_precision=edge["precision"],
            edge_recall=edge["recall"],
            edge_tp=edge["tp"],
            edge_fp=edge["fp"],
            edge_fn=edge["fn"],
        )
    )
    # stash for plotting
    scored[name]["opt"] = node
    scored[name]["opt_edge"] = edge

ops = pd.DataFrame(op_rows).sort_values("discovery_auroc", ascending=False)
print(f"Per-MOCK node metrics at pooled t*={OPT_THRESH:.3f}")
display(
    ops[
        [
            "name",
            "discovery_auroc",
            "discovery_ap",
            "precision",
            "recall",
            "f1",
            "tp",
            "fp",
            "fn",
            "tn",
        ]
    ].round(4)
)
print(f"\nPer-MOCK edge metrics at pooled t*={OPT_EDGE_THRESH:.3f}")
display(
    ops[
        [
            "name",
            "edge_auroc",
            "edge_precision",
            "edge_recall",
            "edge_tp",
            "edge_fp",
            "edge_fn",
        ]
    ].round(4)
)

order = ops.sort_values("discovery_auroc")
ypos = np.arange(len(order))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharey=True)

ax = axes[0]
ax.barh(ypos, order["discovery_auroc"], color="0.75", label="AUROC")
ax.scatter(order["precision"], ypos, color="C3", zorder=3, label=f"P @ t*={OPT_THRESH:.2f}")
ax.scatter(order["recall"], ypos, color="C0", marker="D", s=18, zorder=3, label=f"R @ t*")
ax.scatter(order["f1"], ypos, color="C1", marker="x", s=28, zorder=3, label="F1 @ t*")
ax.set_yticks(ypos)
ax.set_yticklabels(order["name"])
ax.set_xlim(0, 1.02)
ax.axvline(0.5, color="0.7", lw=0.8, ls=":")
ax.set_xlabel("score")
ax.set_title("Node: ranking vs operating point")
ax.legend(loc="lower right", fontsize=7)

ax = axes[1]
ax.barh(ypos, order["edge_auroc"], color="0.75", label="edge AUROC")
ax.scatter(order["edge_precision"], ypos, color="C3", zorder=3, label=f"edge P @ t*")
ax.scatter(order["edge_recall"], ypos, color="C2", marker="s", s=16, zorder=3, label="edge R @ t*")
ax.set_xlim(0, 1.02)
ax.axvline(0.5, color="0.7", lw=0.8, ls=":")
ax.set_xlabel("score")
ax.set_title("Edge: ranking vs operating point")
ax.legend(loc="lower right", fontsize=7)

fig.tight_layout()
fig.savefig(FIG_DIR / "per_mock_at_opt_thresh.pdf")
plt.show()

## Spatial drawings at $t^\star$

Four panels: truth, predicted $p$, discovery on `~train_mask`, and **val∪test only**
TP/FP/FN at `OPT_THRESH` (FP markers are smaller — they usually dominate).

In [ ]:
def _subsample_bg(mask: np.ndarray, max_bg: int, rng: np.random.Generator) -> np.ndarray:
    idx = np.flatnonzero(mask)
    if len(idx) > max_bg:
        idx = rng.choice(idx, size=max_bg, replace=False)
    return idx


def plot_stream_panels(
    result: dict,
    *,
    p_thresh: float = OPT_THRESH,
    max_bg: int = 40_000,
    max_edges: int = 2_500,
    seed: int = 0,
) -> plt.Figure:
    rng = np.random.default_rng(seed)
    phi, lam = result["phi"], result["lam"]
    y = result["y"]
    p = result["p_node"]
    tr = result["train_mask"]
    pool = result["pool"]
    op = result.get("opt") or confusion_at_threshold(p, y, pool, p_thresh)

    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), sharex=True, sharey=True, dpi=800)

    ax = axes[0]
    bg = _subsample_bg(~y, max_bg, rng)
    ax.scatter(phi[bg], lam[bg], s=1.5, c="0.75", alpha=0.25, rasterized=True, linewidths=0)
    ax.scatter(phi[y], lam[y], s=10, c="crimson", alpha=0.9, rasterized=True, linewidths=0)
    pairs = result["pairs"]
    ey = result["edge_y"].astype(bool)
    ss = np.flatnonzero(ey)
    if len(ss) > max_edges:
        ss = rng.choice(ss, size=max_edges, replace=False)
    if len(ss):
        seg = np.stack(
            [
                np.column_stack([phi[pairs[0, ss]], lam[pairs[0, ss]]]),
                np.column_stack([phi[pairs[1, ss]], lam[pairs[1, ss]]]),
            ],
            axis=1,
        )
        ax.add_collection(
            LineCollection(seg, colors="crimson", linewidths=0.35, alpha=0.35, rasterized=True)
        )
    ax.set_title(f"{result['name']} truth\ny+={result['y_pos']}")
    ax.set_xlabel(r"$\phi$")
    ax.set_ylabel(r"$\lambda$")

    ax = axes[1]
    show = np.concatenate([_subsample_bg(~y, max_bg, rng), np.flatnonzero(y)])
    sc = ax.scatter(
        phi[show],
        lam[show],
        c=p[show],
        s=np.where(y[show], 12, 2),
        cmap="magma",
        vmin=0,
        vmax=1,
        alpha=0.85,
        rasterized=True,
        linewidths=0,
    )
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label=r"$p(\mathrm{stream})$")
    auroc = result["discovery_auroc"]
    title = f"predicted\nAUROC={auroc:.3f}" if np.isfinite(auroc) else "predicted"
    #if np.isfinite(result.get("discovery_ap", np.nan)):
        #title += f"  AP={result['discovery_ap']:.3f}"
    ax.set_title(title)
    ax.set_xlabel(r"$\phi$")

    ax = axes[2]
    bg = _subsample_bg(pool & ~y, max_bg, rng)
    ax.scatter(phi[bg], lam[bg], s=1.5, c="0.8", alpha=0.2, rasterized=True, linewidths=0)
    pred = p >= p_thresh
    tp = pool & y & pred
    fn = pool & y & ~pred
    fp = pool & ~y & pred
    ax.scatter(phi[tp], lam[tp], s=12, c="C0", alpha=0.9, rasterized=True, linewidths=0, label="TP")
    ax.scatter(phi[fn], lam[fn], s=14, c="C3", alpha=0.9, rasterized=True, linewidths=0, label="FN")
    ax.scatter(phi[fp], lam[fp], s=8, c="C1", alpha=0.7, rasterized=True, linewidths=0, label="FP")
    ax.scatter(
        phi[tr & y],
        lam[tr & y],
        s=6,
        facecolors="none",
        edgecolors="0.2",
        linewidths=0.4,
        alpha=0.5,
        label="train y=1",
    )
    ax.legend(loc="best", markerscale=1.2, frameon=True)
    ax.set_title(
        f"discovery @ p>={p_thresh:.2f}\n"
        f"P={op['precision']:.2f} R={op['recall']:.2f}\n"
        f"TP={op['tp']} FP={op['fp']} FN={op['fn']}"
    )
    ax.set_xlabel(r"$\phi$")

    for ax in axes:
        ax.set_aspect("equal", adjustable="box")
    fig.suptitle(result["stream_label"], y=1.05)
    fig.tight_layout()
    return fig


print(f"Drawing discovery maps at OPT_THRESH={OPT_THRESH:.3f} (rule={THRESH_RULE})")
for name in STREAMS_TO_DRAW:
    if name not in scored:
        print(f"scoring missing {name}")
        scored[name] = score_stream(name)
        scored[name]["opt"] = confusion_at_threshold(
            scored[name]["p_node"], scored[name]["y"], scored[name]["pool"], OPT_THRESH
        )
    fig = plot_stream_panels(scored[name], p_thresh=OPT_THRESH)
    fig.savefig(FIG_DIR / f"{name}_panels_tstar.pdf", bbox_inches="tight")
    plt.show()

## Quick read

- Change `THRESH_RULE` (`f1` / `fbeta` / `prec_at_recall`) and re-run the threshold cells +
  drawings — maps follow `OPT_THRESH`.
- If AUROC is high but P@$t^*$ is low, the ranking is fine and the cut is the problem:
  raise $t$ or switch to `prec_at_recall`.
- `$p=0.5$` is only a reference line; with ~0.1% positives it is usually far too loose.